# 更正data中的特征


In [1]:
import pandas as pd

# 读取两个表格
train_set_new = pd.read_excel('train_set_new.xlsx', index_col=0)
full_8 = pd.read_excel('data/FULL_8.xlsx', index_col=0)

# 获取列名
train_columns = set(train_set_new.columns)
full_8_columns = set(full_8.columns)

# 找出不同的列名
unique_to_train_set_new = train_columns - full_8_columns
unique_to_full_8 = full_8_columns - train_columns

# 打印不同的列名
print("Columns in train_set_new but not in FULL_8:")
for col in unique_to_train_set_new:
    print(col)

print("\nColumns in FULL_8 but not in train_set_new:")
for col in unique_to_full_8:
    print(col)


Columns in train_set_new but not in FULL_8:
Yang omega
Interant electrons
Radii local mismatch
Interant p electrons
Radii gamma
Total weight
VEC mean
Precipitate Distribution
Yang delta
APE mean
Electronegativity delta
Habit Plane
Shear modulus local mismatch
Configuration entropy
Electronegativity local mismatch
Interant s electrons
Shear modulus strength model
Shear modulus mean
Atomic weight mean
Mean cohesive energy
Shear modulus delta
Interant d electrons
Mixing enthalpy
Interant f electrons
Lambda entropy

Columns in FULL_8 but not in train_set_new:
MagpieData maximum MeltingT
MagpieData range MeltingT
MagpieData minimum MeltingT


In [3]:
import os
import pandas as pd

# 读取train_set_new.xlsx表格
train_set_new = pd.read_excel('data/train_set_new.xlsx', index_col=0)

# 指定要删除的列
columns_to_drop = [
    'MagpieData maximum MeltingT',
    'MagpieData range MeltingT',
    'MagpieData minimum MeltingT'
]

# 获取data文件夹下所有以FULL开头的xlsx文件
data_folder = 'data'
files = [f for f in os.listdir(data_folder) if f.startswith('FULL') and f.endswith('.xlsx')]

# 处理每个文件
for file in files:
    file_path = os.path.join(data_folder, file)
    
    # 读取文件
    df = pd.read_excel(file_path, index_col=0)
    
    # 删除屈服强度为0的行
    df = df.dropna(subset=['屈服强度'])
    
    # 删除指定的列
    df.drop(columns=columns_to_drop, inplace=True, errors='ignore')
    
    # 替换同名列的数据
    common_columns = df.columns.intersection(train_set_new.columns)
    df[common_columns] = train_set_new[common_columns]
    
    # 如果文件名包含Mor或者mor，在倒数第三列之前添加Habit Plane列
    if 'Mor' in file or 'mor' in file:
        habit_plane_col = df['Habit Plane']  # 获取'Habit Plane'列的数据
        df.drop(columns=['Habit Plane'], inplace=True)  # 删除原来的'Habit Plane'列
        position = df.shape[1] - 3  # 倒数第三列的位置
        df.insert(position, 'Habit Plane', habit_plane_col)  # 将'Habit Plane'列插入到指定位置

    # 保存修改后的文件
    df.to_excel(file_path)

print("All files have been processed.")


C:\Users\acer-pc\AppData\Local\Temp\ipykernel_2628\3534614340.py:40: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df.insert(position, 'Habit Plane', habit_plane_col)  # 将'Habit Plane'列插入到指定位置
C:\Users\acer-pc\AppData\Local\Temp\ipykernel_2628\3534614340.py:40: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df.insert(position, 'Habit Plane', habit_plane_col)  # 将'Habit Plane'列插入到指定位置
C:\Users\acer-pc\AppData\Local\Temp\ipykernel_2628\3534614340.py:40: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result

All files have been processed.


#仅采用Elements数据进行预测

In [2]:
# # K折交叉验证的模型结果
# from sklearn.model_selection import GridSearchCV, KFold
# from sklearn.metrics import r2_score, mean_absolute_error, mean_absolute_percentage_error
# import numpy as np
# import pandas as pd
# import joblib
# from openpyxl import load_workbook
# from sklearn.model_selection import train_test_split
# 
# # 定义文件夹路径
# folder_path_qf = "models/qf"
# folder_path_kl = 'models/kl'
# 
# # 更改数据来源和保存文件夹
# data_path = 'data/FULL_elements.xlsx'
# save_path = 'individual_models_scores_elements1.xlsx' 
# 
# feature_name = ['Al', 'Zn', 'Mn', 'Mg', 'Li', 'Sn', 'Ce', 'La', 'Ca', 'Si', 'Cu', 'Ni',
#   'Fe', 'Bi', 'Ti', 'Sr', 'Sm', 'Sc', 'Gd', 'Y', 'Zr', 'Nd', 'Ag', 'Yb']
# 
# # 定义模型权重
# weights_qf = {'gboost': 0.5/5, 'rf': 0.5/5}  # 注意所有模型系数和为1
# weights_kl = {'gboost': 0.4/5,  'xgboost': 0.6/5}
# 
# # 初始化列表以存储模型评估指标
# model_scores_qf = []
# model_scores_kl = []
# 
# # 读取数据
# df_qf = pd.read_excel(data_path, index_col=0)
# df_kl = pd.read_excel(data_path, index_col=0)
# 
# # 删除包含空值的行
# df_qf = df_qf[df_qf['屈服强度'] != 0].reset_index(drop=True)
# df_kl = df_kl[df_kl['抗拉强度 （UTS）'] != 0].reset_index(drop=True)
# 
# # 分割特征和目标变量
# X_qf = df_qf[feature_name]
# y_qf = df_qf['屈服强度']
# X_kl = df_kl[feature_name]
# y_kl = df_kl['抗拉强度 （UTS）']
# 
# # 定义五折交叉验证
# kf = KFold(n_splits=5, shuffle=True, random_state=200)
# 
# # 初始化列表以存储每个折的评估指标
# r2_scores_qf = []
# mae_scores_qf = []
# mape_scores_qf = []
# r2_scores_kl = []
# mae_scores_kl = []
# mape_scores_kl = []
# 
# # 在每个折上进行模型训练和预测（QF问题）
# for train_index, test_index in kf.split(X_qf):
#     X_train_qf, X_test_qf = X_qf.iloc[train_index], X_qf.iloc[test_index]
#     y_train_qf, y_test_qf = y_qf.iloc[train_index], y_qf.iloc[test_index]
# 
#     # 初始化列表以存储加权结果
#     predictions_fold_qf = np.zeros(len(y_test_qf))
# 
#     # 遍历QF模型进行加权
#     for model_name, weight in weights_qf.items():
#         for i in range(1, 6):
#             model_path = f'{folder_path_qf}/{model_name}/{model_name}{i}.pkl'
#             model = joblib.load(model_path)
#             predictions_fold_qf += model.predict(X_test_qf) * weight
#     
#     # 计算折的评估指标
#     r2_fold_qf = r2_score(y_test_qf, predictions_fold_qf)
#     mae_fold_qf = mean_absolute_error(y_test_qf, predictions_fold_qf)
#     mape_fold_qf = mean_absolute_percentage_error(y_test_qf, predictions_fold_qf)
# 
#     # 添加到列表中
#     r2_scores_qf.append(r2_fold_qf)
#     mae_scores_qf.append(mae_fold_qf)
#     mape_scores_qf.append(mape_fold_qf)
# 
# # 在每个折上进行模型训练和预测（KL问题）
# for train_index, test_index in kf.split(X_kl):
#     X_train_kl, X_test_kl = X_kl.iloc[train_index], X_kl.iloc[test_index]
#     y_train_kl, y_test_kl = y_kl.iloc[train_index], y_kl.iloc[test_index]
# 
#     # 初始化列表以存储加权结果
#     predictions_fold_kl = np.zeros(len(y_test_kl))
# 
#     # 遍历KL模型进行加权
#     for model_name, weight in weights_kl.items():
#         for i in range(1, 6):
#             model_path = f'{folder_path_kl}/{model_name}/{model_name}{i}.pkl'
#             model = joblib.load(model_path)
#             predictions_fold_kl += model.predict(X_test_kl) * weight
#     
#     # 计算折的评估指标
#     r2_fold_kl = r2_score(y_test_kl, predictions_fold_kl)
#     mae_fold_kl = mean_absolute_error(y_test_kl, predictions_fold_kl)
#     mape_fold_kl = mean_absolute_percentage_error(y_test_kl, predictions_fold_kl)
# 
#     # 添加到列表中
#     r2_scores_kl.append(r2_fold_kl)
#     mae_scores_kl.append(mae_fold_kl)
#     mape_scores_kl.append(mape_fold_kl)
# 
# # 计算五折交叉验证的平均指标（QF问题）
# r2_final_qf = np.mean(r2_scores_qf)
# mae_final_qf = np.mean(mae_scores_qf)
# mape_final_qf = np.mean(mape_scores_qf)
# 
# # 计算五折交叉验证的平均指标（KL问题）
# r2_final_kl = np.mean(r2_scores_kl)
# mae_final_kl = np.mean(mae_scores_kl)
# mape_final_kl = np.mean(mape_scores_kl)
# 
# # 将最终模型的评估指标添加到列表中（QF问题）
# model_scores_qf.append({
#     "Model": "Final",  # e.g., gboost_1
#     "R^2": r2_final_qf,
#     "MAE": mae_final_qf,
#     "MAPE": mape_final_qf
# })
# 
# # 将最终模型的评估指标添加到列表中（KL问题）
# model_scores_kl.append({
#     "Model": "Final",  # e.g., gboost_1
#     "R^2": r2_final_kl,
#     "MAE": mae_final_kl,
#     "MAPE": mape_final_kl
# })
# 
# # 将列表转换为DataFrame以便于查看（QF问题）
# scores_df_qf = pd.DataFrame(model_scores_qf)
# 
# # 将列表转换为DataFrame以便于查看（KL问题）
# scores_df_kl = pd.DataFrame(model_scores_kl)
# 
# from openpyxl.utils.dataframe import dataframe_to_rows
# 
# # 将DataFrame写入到指定工作表名的Excel文件中（附加到现有工作表）
# def write_to_existing_excel(scores_df, file_path, sheet_name):
#     try:
#         wb = load_workbook(file_path)
#         ws = wb[sheet_name]
#         # 计算新数据的起始行
#         start_row = ws.max_row + 1
#         # 将数据写入工作表
#         for row in dataframe_to_rows(scores_df, index=False, header=True):
#             ws.append(row)
#         # 保存工作表
#         wb.save(file_path)
#     except FileNotFoundError:
#         # 如果文件不存在，则创建新文件并将数据写入
#         scores_df.to_excel(file_path, index=False, sheet_name=sheet_name)
# 
# # 调用函数将数据写入Excel文件
# write_to_existing_excel(scores_df_qf, save_path, 'qf_final')
# write_to_existing_excel(scores_df_kl, save_path, 'kl_final')


ValueError: The feature names should match those that were passed during fit.
Feature names seen at fit time, yet now missing:
- APE mean
- Ac
- Ac fraction
- Ag fraction
- Al fraction
- ...


计算单个数据，单个预测目标

In [5]:
import os.path
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import r2_score, mean_absolute_percentage_error,mean_absolute_error
from utils import create_empty_ls,create_dir,mycopyfile,mycopydir
import joblib
from openpyxl import load_workbook
from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import GradientBoostingRegressor  # 导入GradientBoostingRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import AdaBoostRegressor
from sklearn.neighbors import KNeighborsRegressor
save_model=True
del_0qf=True #是否删除0屈服强度数据
save_index=True
Regressor=GradientBoostingRegressor
# Regressor=XGBRegressor
# Regressor=RandomForestRegressor

label='kl'
data_name = 'FULL_elements'
if not os.path.exists(data_name):
    os.mkdir(data_name)
else:
    print(f'{data_name}文件夹已存在')

main_path=f'{data_name}/models/{label}'  #  研究抗拉强度
model_name1 = 'gboost'
# 这里记录着分折的结果,方便后续读取序列作shap分析
index_dir=f'{data_name}/index'

save_path = f"{data_name}/individual_model_scores_{data_name}.xlsx"

data_path = f'data/{data_name}.xlsx'
feature_names = ['Al', 'Zn', 'Mn', 'Mg', 'Li', 'Sn', 'Ce', 'La', 'Ca', 'Si', 'Cu', 'Ni',
  'Fe', 'Bi', 'Ti', 'Sr', 'Sm', 'Sc', 'Gd', 'Y', 'Zr', 'Nd', 'Ag', 'Yb']

if not os.path.exists(main_path):
    os.mkdir(f'{data_name}/models')
    os.makedirs(main_path)
    print(f'创建{main_path}文件夹')
else:
    pass
if not os.path.exists(index_dir):
    print('创建index文件夹')
    os.mkdir(index_dir)
else:
    print('index文件夹已存在')
    
index_excel=f'{index_dir}/{label}_index.xlsx'

if not os.path.exists(index_excel):
    print('生成index文件')
    df=pd.DataFrame()
    df.to_excel(index_excel)
else:
    print('index文件已存在')


n_splits=5
ls1=['fold'+str(i) for i in range(1,n_splits+1)]
fold_dict=dict(zip(ls1,[0 for i in range(0,len(ls1))]))
# 设置随机种子以确保结果的可重复性
np.random.seed(200)

params = {
    'alpha': 0.9,
    'ccp_alpha': 0.0,
    'criterion': 'friedman_mse',
    'init': None,
    'learning_rate': 0.1,
    'loss': 'squared_error',  # 'squared_error'在GradientBoostingRegressor中对应'ls'
    'max_depth': 3,
    'max_features': None,
    'max_leaf_nodes': None,
    'min_impurity_decrease': 0.0,
    'min_samples_leaf': 1,
    'min_samples_split': 2,
    'min_weight_fraction_leaf': 0.0,
    'n_estimators': 100,
    'n_iter_no_change': None,
    'random_state': 200,
    'subsample': 1.0,
    'tol': 0.0001,
    'validation_fraction': 0.1,
    'verbose': 0,
    'warm_start': False
}

# 读取 Excel 文件
df = pd.read_excel(data_path, index_col=0)  # 引入这一列之后，原本的第一列就是一个索引，第0列会从有意义的列开始
# 删除包含空值的行
if del_0qf:
    df = df[df['抗拉强度 （UTS）'] != 0].reset_index(drop=True)
X = df[feature_names]  # 特征: 最后四列之前的所有列
y = df['抗拉强度 （UTS）']  # 目标: 倒数第四列
# print('X.columns:\n',X.columns)
# print('特征的形状:\n',len(feature_names))
# print('输入数据的形状：\n',X.shape)

# 将目标变量分箱为10个类别有利于进一步分折（但是预测标签不会这个改变）
bins = np.linspace(y.min(), y.max(), 11)
y_binned = np.digitize(y, bins)

# 初始化StratifiedKFold
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=200)

model= Regressor(**params)  # 如果这里固定初始化模型删除掉的话可以在下面循环呢内添加随机初始化或者指定个别参数

# 初始化列表以存储每个折叠的分数
r2_scores = []
mape_scores = []
mae_scores = []
model_scores = []  # 存储每个分折的评估指标

index = 0
num_each_fold_train = np.floor((np.int32(X.shape[0]) - 1) * (n_splits - 1) / n_splits)
num_each_fold_test = (np.int32(X.shape[0]) - 1) // n_splits
for train_index, test_index in skf.split(X, y_binned):
    X_train, X_test = X.iloc[train_index[:int(num_each_fold_train)]], X.iloc[test_index[:int(num_each_fold_test)]]
    y_train, y_test = y.iloc[train_index[:int(num_each_fold_train)]], y.iloc[test_index[:int(num_each_fold_test)]]
    index = index + 1
    fold_dict['fold' + str(index)] = train_index[:int(num_each_fold_train)].astype(int)  # 记录训练集索引    # print(f'fold_{index}',fold_dict)
    # 训练模型
    model.fit(X_train, y_train)
    if not os.path.exists(f'{main_path}/{model_name1}'):
        os.mkdir(f'{main_path}/{model_name1}')
    joblib.dump(model,   f'{main_path}/{model_name1}/{model_name1}{index}.pkl')
    print('保存模型文件')

    # 在测试集上进行预测
    y_pred = model.predict(X_test)

    # 计算并存储分数
    r2 = r2_score(y_test, y_pred)
    mape = mean_absolute_percentage_error(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    r2_scores.append(r2)
    mape_scores.append(mape)
    mae_scores.append(mae)

    # 存储每个分折的评估指标到字典中
    model_scores.append({
        "Fold": index,
        "R^2": r2,
        "MAPE": mape,
        "MAE": mae
    })


ls = list(fold_dict.values())
# print('ls',ls)
df = pd.DataFrame(ls)

book = load_workbook(index_excel)
with pd.ExcelWriter(index_excel, engine='openpyxl') as writer:
    writer.book = book
    df.to_excel(writer, sheet_name='k-fold')

# 计算最终模型的性能
final_r2 = np.mean(r2_scores)
final_mape = np.mean(mape_scores)
final_mae = np.mean(mae_scores)

# 添加最终模型性能到DataFrame中
model_scores.append({
    "Fold": "Final",
    "R^2": final_r2,
    "MAPE": final_mape,
    "MAE": final_mae
})

# 创建DataFrame来存储每个分折的评估指标和最终模型性能
model_scores_df = pd.DataFrame(model_scores)

if not os.path.exists(save_path):
    df = pd.DataFrame()
    df.to_excel(save_path, index=False)

from openpyxl import load_workbook
from openpyxl.utils.dataframe import dataframe_to_rows

def write_to_existing_excel(scores_df, file_path, sheet_name):
    try:
        wb = load_workbook(file_path)
        if sheet_name in wb.sheetnames:  # 检查工作表是否存在
            ws = wb[sheet_name]
        else:
            ws = wb.create_sheet(sheet_name)  # 创建一个新的工作表
        start_row = ws.max_row + 1
        for r in dataframe_to_rows(scores_df, index=False, header=False):
            ws.append(r)
        wb.save(file_path)
    except FileNotFoundError:
        scores_df.to_excel(file_path, index=False, sheet_name=sheet_name)

write_to_existing_excel(model_scores_df, save_path, 'kl_gboost')



FULL_elements文件夹已存在
index文件夹已存在
index文件已存在
保存模型文件
保存模型文件
保存模型文件
保存模型文件
保存模型文件


计算单个任务，两个预测目标

In [11]:
import os.path
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import r2_score, mean_absolute_percentage_error, mean_absolute_error
import joblib
from openpyxl import load_workbook
from sklearn.ensemble import GradientBoostingRegressor
from openpyxl.utils.dataframe import dataframe_to_rows

# 设置模型参数和全局配置
save_model = True
del_0qf = True  # 是否删除0屈服强度数据
save_index = True
Regressor = GradientBoostingRegressor
label = 'kl'
data_name = 'FULL_elements1'
feature_names = ['Al', 'Zn', 'Mn', 'Mg', 'Li', 'Sn', 'Ce', 'La', 'Ca', 'Si', 'Cu', 'Ni',
                 'Fe', 'Bi', 'Ti', 'Sr', 'Sm', 'Sc', 'Gd', 'Y', 'Zr', 'Nd', 'Ag', 'Yb']
# 确保文件夹存在
if not os.path.exists(data_name):
    os.mkdir(data_name)
else:
    print(f'{data_name}文件夹已存在')

main_path = f'{data_name}/models/{label}'
model_name1 = 'gboost'
index_dir = f'{data_name}/index'
save_path = f"{data_name}/individual_model_scores_{data_name}.xlsx"
data_path = f'data/{data_name}.xlsx'


# 创建必要的目录
if not os.path.exists(main_path):
    os.makedirs(f'{data_name}/models',exist_ok = True)
    os.makedirs(main_path,exist_ok = True)
    print(f'创建{main_path}文件夹')
if not os.path.exists(index_dir):
    os.mkdir(index_dir)
    print('创建index文件夹')
else:
    print('index文件夹已存在')

index_excel = f'{index_dir}/{label}_index.xlsx'
if not os.path.exists(index_excel):
    print('生成index文件')
    pd.DataFrame().to_excel(index_excel)
else:
    print('index文件已存在')

# 读取 Excel 文件
df = pd.read_excel(data_path, index_col=0)

# 分别处理两个目标变量
df_kl = df[df['抗拉强度 （UTS）'] != 0].reset_index(drop=True)
df_qf = df[df['屈服强度'] != 0].reset_index(drop=True)

X_kl = df_kl[feature_names]
y_kl = df_kl['抗拉强度 （UTS）']
X_qf = df_qf[feature_names]
y_qf = df_qf['屈服强度']

# 初始化StratifiedKFold
skf_kl = StratifiedKFold(n_splits=5, shuffle=True, random_state=200)
skf_qf = StratifiedKFold(n_splits=5, shuffle=True, random_state=200)

params = {
    'alpha': 0.9,
    'ccp_alpha': 0.0,
    'criterion': 'friedman_mse',
    'init': None,
    'learning_rate': 0.1,
    'loss': 'squared_error',  # 'squared_error'在GradientBoostingRegressor中对应'ls'
    'max_depth': 3,
    'max_features': None,
    'max_leaf_nodes': None,
    'min_impurity_decrease': 0.0,
    'min_samples_leaf': 1,
    'min_samples_split': 2,
    'min_weight_fraction_leaf': 0.0,
    'n_estimators': 100,
    'n_iter_no_change': None,
    'random_state': 200,
    'subsample': 1.0,
    'tol': 0.0001,
    'validation_fraction': 0.1,
    'verbose': 0,
    'warm_start': False
}

# 初始化模型
model_kl = Regressor(**params)
model_qf = Regressor(**params)
# bins = np.linspace(y.min(), y.max(), 11)
# y_binned = np.digitize(y, bins)

bins_qf = np.linspace(y_qf.min(), y_qf.max(), 11)
y_binned_qf = np.digitize(y_qf, bins_qf)

bins_kl = np.linspace(y_kl.min(), y_kl.max(), 11)
y_binned_kl = np.digitize(y_kl, bins_kl)
# 训练和评估模型
def train_and_evaluate(X, y,y_binned, model, skf, label):
    scores = []
    

    for fold, (train_index, test_index) in enumerate(skf.split(X, y_binned)):
        X_train, X_test = X.iloc[train_index], X.iloc[test_index]
        y_train, y_test = y.iloc[train_index], y.iloc[test_index]
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        scores.append({
            "Fold": fold + 1,
            "R^2": r2_score(y_test, y_pred),
            "MAPE": mean_absolute_percentage_error(y_test, y_pred),
            "MAE": mean_absolute_error(y_test, y_pred)
        })
        if save_model:
            model_path = f'{main_path}/{model_name1}/{label}_model{fold+1}.pkl'
            os.makedirs(os.path.dirname(model_path), exist_ok=True)
            joblib.dump(model, model_path)
    return scores

scores_kl = train_and_evaluate(X_kl, y_kl,y_binned_kl, model_kl, skf_kl, 'kl')
scores_qf = train_and_evaluate(X_qf, y_qf, y_binned_qf,model_qf, skf_qf, 'qf')

def write_to_existing_excel(scores_df, file_path, sheet_name):
    try:
        wb = load_workbook(file_path)
        if sheet_name in wb.sheetnames:  # 检查工作表是否存在
            ws = wb[sheet_name]
        else:
            ws = wb.create_sheet(sheet_name)  # 创建一个新的工作表
        start_row = ws.max_row + 1
        for r in dataframe_to_rows(scores_df, index=False, header=False):
            ws.append(r)
        wb.save(file_path)
    except FileNotFoundError:
        scores_df.to_excel(file_path, index=False, sheet_name=sheet_name)
        
# 保存评估结果到Excel
def save_scores(scores, sheet_name):
    scores_df = pd.DataFrame(scores)
    
    # 计算平均分数
    final_scores = {
        "Fold": "Final",
        "R^2": scores_df["R^2"].mean(),
        "MAPE": scores_df["MAPE"].mean(),
        "MAE": scores_df["MAE"].mean()
    }
    
    # 将最终分数添加到DataFrame
    scores_df = scores_df.append(final_scores, ignore_index=True)
    
    # 写入到Excel
    write_to_existing_excel(scores_df, save_path, sheet_name)

save_scores(scores_kl, 'kl_gboost')
save_scores(scores_qf, 'qf_gboost')


FULL_elements1文件夹已存在
创建FULL_elements1/models/kl文件夹
index文件夹已存在
生成index文件
